In [34]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.decomposition import PCA
from unidecode import unidecode
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
import json
from unidecode import unidecode
import requests
import re

In [35]:
def build_stats(filename_full='stats_full', filename_reduced='stats_reduced'):
    df = pd.read_csv('unified_player_stats.csv')
    df = df.loc[df['season'] == "2025-26"]
    df = df[['player', '_name_norm', 'team', 'pos', 'games', 'minutes', 'ninety_s', 'shots', 'shots_per90', 'shots_on_target', 'shots_inside_box', 'shots_outside_box', 'npxg', 'npxg_per90', 'npxg_overperformance', 'goal_conversion_pct', 'big_chances_missed', 'key_passes_per90', 'xag_per90', 'attempt_assists', 'big_chances_created_per90', 'xg_chain_per90', 'xg_buildup_per90', 'pass_to_assist', 'passes_total', 'pass_completion_pct', 'passes_final_third', 'passes_opp_half', 'long_balls_total', 'long_balls_pct', 'crosses_total', 'crosses_pct', 'chipped_passes_total', 'chipped_passes_pct', 'touches', 'dribbles_per90', 'dribbles_pct', 'dispossessed', 'possession_lost', 'ball_recoveries', 'possession_won_att_third', 'tackles', 'tackles_won_pct', 'interceptions_per90', 'clearances', 'blocked_shots', 'dribbled_past', 'aerials_won_pct', 'ground_duels_won_pct', 'duels_won_pct', 'fouls', 'fouled']]
    df['shots_on_target_per90'] = df['shots_on_target']/df['ninety_s']
    df['shots_on_target_pct'] = df['shots_on_target']/df['shots']
    df["shots_on_target_pct"] = df["shots_on_target_pct"].replace([np.inf, -np.inf], 0)
    df["shots_on_target_pct"] = df["shots_on_target_pct"].fillna(0)
    df['shots_inside_box_per90'] = df['shots_inside_box']/df['ninety_s']
    df['shots_outside_box_per90'] = df['shots_outside_box']/df['ninety_s']
    df['big_chances_missed_per90'] = df['big_chances_missed']/df['ninety_s']
    df['attempt_assists_per90'] = df['attempt_assists']/df['ninety_s']
    df['pass_to_assist_per90'] = df['pass_to_assist']/df['ninety_s']
    df['passes_per90'] = df['passes_total']/df['ninety_s']
    df['passes_final_third_per90'] = df['passes_final_third']/df['ninety_s']
    df['passes_opp_half_per90'] = df['passes_opp_half']/df['ninety_s']
    df['long_balls_per90'] = df['long_balls_total']/df['ninety_s']
    df['crosses_per90'] = df['crosses_total']/df['ninety_s']
    df['chipped_passes_per90'] = df['chipped_passes_total']/df['ninety_s']
    df['touches_per90'] = df['touches']/df['ninety_s']
    df['dispossessed_per90'] = df['dispossessed']/df['ninety_s']
    df['possession_lost_per90'] = df['possession_lost']/df['ninety_s']
    df['ball_recoveries_per90'] = df['ball_recoveries']/df['ninety_s']
    df['possession_won_att_third_per90'] = df['possession_won_att_third']/df['ninety_s']
    df['tackles_per90'] = df['tackles']/df['ninety_s']
    df['clearances_per90'] = df['clearances']/df['ninety_s']
    df['blocked_shots_per90'] = df['blocked_shots']/df['ninety_s']
    df['dribbled_past_per90'] = df['dribbled_past']/df['ninety_s']
    df['fouls_per90'] = df['fouls']/df['ninety_s']
    df['fouled_per90'] = df['fouled']/df['ninety_s']
    df['npxg_overperformance_per90'] = df['npxg_overperformance']/df['ninety_s']
    df['npxg_per_shot'] = df['npxg']/df['shots']
    df["npxg_per_shot"] = df["npxg_per_shot"].fillna(0)
    # df['player_norm'] = df['_name_norm']
    df['player_norm'] = df['player'].apply(unidecode)

    df.to_csv(f"{filename_full}.csv", index=False, header=True)
    df_reduced = df[['player', 'player_norm', 'team', 'pos', 'games', 'minutes', 'ninety_s', 'shots_per90', 'shots_on_target_per90', 'shots_on_target_pct', 'shots_inside_box_per90', 'shots_outside_box_per90', 'npxg_per_shot', 'npxg_per90', 'npxg_overperformance_per90', 'goal_conversion_pct', 'big_chances_missed_per90', 'key_passes_per90', 'xag_per90', 'attempt_assists_per90', 'big_chances_created_per90', 'xg_chain_per90', 'xg_buildup_per90', 'passes_per90', 'pass_completion_pct', 'passes_final_third_per90', 'long_balls_per90', 'long_balls_pct', 'crosses_per90', 'crosses_pct', 'chipped_passes_per90', 'chipped_passes_pct', 'touches_per90', 'dribbles_per90', 'dribbles_pct', 'dispossessed_per90', 'possession_lost_per90', 'ball_recoveries_per90', 'possession_won_att_third_per90', 'tackles_per90', 'tackles_won_pct', 'interceptions_per90', 'clearances_per90', 'blocked_shots_per90', 'dribbled_past_per90', 'aerials_won_pct', 'ground_duels_won_pct', 'duels_won_pct', 'fouls_per90', 'fouled_per90']]
    df_reduced.to_csv(f"{filename_reduced}.csv", index=False, header=True)

In [36]:
def build_df(filename='stats_reduced'):

    df_min = pd.read_csv('stats_reduced.csv')
    df_min['goal_conversion_pct'] = df_min['goal_conversion_pct']/100
    df_min['pass_completion_pct'] = df_min['pass_completion_pct']/100
    df_min['long_balls_pct'] = df_min['long_balls_pct']/100
    df_min['crosses_pct'] = df_min['crosses_pct']/100
    df_min['chipped_passes_pct'] = df_min['chipped_passes_pct']/100
    df_min['dribbles_pct'] = df_min['dribbles_pct']/100
    df_min['tackles_won_pct'] = df_min['tackles_won_pct']/100
    df_min['aerials_won_pct'] = df_min['aerials_won_pct']/100
    df_min['ground_duels_won_pct'] = df_min['ground_duels_won_pct']/100
    df_min['duels_won_pct'] = df_min['duels_won_pct']/100


    return df_min

In [37]:
def standardize_df(df, sofa_under=False, minutes=1000):
    if sofa_under:
        df = df.loc[df['position'] != 'GK']
        df = df.loc[df['minutes'] >= 1000]
    else:
        df = df.loc[df['minutes'] >= minutes]
        df = df.loc[(df['pos'] != 'GK') & (df['pos'] != 'GK S')]
        df = df.dropna(subset=['pos'])
    df = df.reset_index(drop=True)
    scaler = preprocessing.StandardScaler()
    if sofa_under:
        data_df = df.drop(columns=['player', 'league', 'team', 'position', 'games', 'minutes', 'nineties'])
    else:
        data_df = df.drop(columns=['player', 'player_norm', 'team', 'pos', 'minutes', 'ninety_s', 'games'])
    standard_df = scaler.fit_transform(data_df)
    standard_df = pd.DataFrame(standard_df, columns=data_df.columns)
    if sofa_under:
        standard_df.insert(0, 'player', df['player'])
        standard_df.insert(0, 'league', df['league'])
        standard_df.insert(2, 'team', df['team'])
        standard_df.insert(3, 'position', df['position'])
        standard_df.insert(4, 'games', df['games'])
        standard_df.insert(5, 'minutes', df['minutes'])
        standard_df.insert(6, 'nineties', df['nineties'])
    else:
        standard_df.insert(0, 'player', df['player'])
        standard_df.insert(0, 'player_norm', df['player_norm'])
        standard_df.insert(2, 'team', df['team'])
        standard_df.insert(3, 'pos', df['pos'])
        standard_df.insert(4, 'minutes', df['minutes'])
        standard_df.insert(5, 'ninety_s', df['ninety_s'])
        standard_df.insert(6, 'games', df['games'])
    return standard_df

In [38]:
def apply_pca(df_standard, sofa_under=False):
    if sofa_under:
        df_pca = df_standard.drop(columns=['player', 'league', 'team', 'position', 'games', 'minutes', 'nineties'])
    else:
        df_pca = df_standard.drop(columns=['player', 'player_norm', 'team', 'pos', 'minutes', 'ninety_s', 'games'])
    pca = PCA(n_components=7)
    data_pca = pca.fit_transform(df_pca)
    data_pca = pd.DataFrame(data_pca)
    if sofa_under:
        data_pca.insert(0, 'player', df_standard['player'])
        data_pca.insert(1, 'league', df_standard['league'])
        data_pca.insert(2, 'team', df_standard['team'])
        data_pca.insert(3, 'position', df_standard['position'])
        data_pca.insert(4, 'games', df_standard['games'])
        data_pca.insert(5, 'minutes', df_standard['minutes'])
        data_pca.insert(6, 'nineties', df_standard['nineties'])
    else:
        data_pca.insert(0, 'player', df_standard['player'])
        data_pca.insert(1, 'player_norm', df_standard['player_norm'])
        data_pca.insert(2, 'team', df_standard['team'])
        data_pca.insert(3, 'pos', df_standard['pos'])
        data_pca.insert(4, 'minutes', df_standard['minutes'])
        data_pca.insert(5, 'ninety_s', df_standard['ninety_s'])
        data_pca.insert(6, 'games', df_standard['games'])

    cont_df = pd.DataFrame()
    feature_names = df_pca.columns
    cont0 = []
    cont1 = []
    cont2 = []
    cont3 = []
    cont4 = []
    cont5 = []
    cont6 = []
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    for i in range(len(feature_names)):
        # print(f"Feature: {feature_names[i]}, Loadings: {loadings[i]}")
        cont0.append(loadings[i][0])
        cont1.append(loadings[i][1])
        cont2.append(loadings[i][2])
        cont3.append(loadings[i][3])
        cont4.append(loadings[i][4])
        cont5.append(loadings[i][5])
        cont6.append(loadings[i][6])
    cont_df['feature'] = feature_names
    cont_df['cont_0'] = cont0
    cont_df['cont_1'] = cont1
    cont_df['cont_2'] = cont2
    cont_df['cont_3'] = cont3
    cont_df['cont_4'] = cont4
    cont_df['cont_5'] = cont5
    cont_df['cont_6'] = cont6
    print(f"PCA Explained Variance: {pca.explained_variance_ratio_}")
    print(f"PCA Explained Variance Sum: {pca.explained_variance_ratio_.sum():.2%}")


    loading_matrix = pd.DataFrame(
        pca.components_.T,
        index=feature_names,
        columns=[
            'PC1', 'PC2', 'PC3',
            'PC4', 'PC5', 'PC6', 'PC7'
        ]
    )

    loading_matrix['loading_strength'] = np.sqrt(
        (loading_matrix ** 2).sum(axis=1)
    )

    loading_matrix.sort_values(
        'loading_strength',
        ascending=False
    )
    return data_pca, cont_df, loading_matrix

In [39]:
def get_distances(player_name, df_pca, sofa_under=False):
    
    if sofa_under:
        df_pca['player'] = df_pca['player'].apply(unidecode)
        search_row = df_pca.loc[df_pca['player'] == player_name]
    else:
        search_row = df_pca.loc[df_pca['player_norm'] == player_name]
    dist_df = pd.DataFrame()
    players = []
    distances = []
    cos_sims = []
    teams = []
    positions = []

    

    for i in tqdm(df_pca.index, "Computing distances"):
        compare_row = df_pca.loc[i]
        if sofa_under:
            search_player = search_row['player']
            compare_player = compare_row['player']
            compare_team = compare_row['team']
            compare_pos = compare_row['position']
        else:
            search_player = search_row['player_norm']
            compare_player = compare_row['player_norm']
            compare_team = compare_row['team']
            compare_pos = compare_row['pos']
        search_data = search_row[[0, 1, 2, 3, 4, 5, 6]].to_numpy(dtype=float)
        compare_data = compare_row[[0, 1, 2, 3, 4, 5, 6]].to_numpy(dtype=float)
        distance = np.linalg.norm(search_data - compare_data)
        cos_sim = cosine_similarity(search_data.reshape(1, -1), compare_data.reshape(1, -1))[0][0]
        # print(f"Distance between {search_player} and {compare_player}: {distance}")
        players.append(compare_player)
        distances.append(distance)
        cos_sims.append(cos_sim)
        teams.append(compare_team)
        positions.append(compare_pos)

    
    
    dist_df['player'] = players
    dist_df['team'] = teams
    dist_df['position'] = positions
    dist_df['distance'] = distances
    dist_df['cosine_similarity'] = cos_sims
    dist_df = dist_df.sort_values(by=['distance'], ascending=True)
    dist_df = dist_df.reset_index(drop=True)
    dist_df['rank_distance'] = dist_df['distance'].rank(method='min')
    dist_df['rank_cosine_similarity'] = dist_df['cosine_similarity'].rank(ascending=False, method='min')
    dist_df['rank_distance'] = dist_df['rank_distance'].astype(int)
    dist_df['rank_cosine_similarity'] = dist_df['rank_cosine_similarity'].astype(int)

    if sofa_under==True:
        search_pos = search_row['position'].iloc[0]
        # print(search_pos, type(search_pos))
        dist_df_pos = dist_df.loc[dist_df['position'] == search_pos]
        dist_df_pos['rank_distance'] = dist_df_pos['distance'].rank(method='min')
        dist_df_pos['rank_cosine_similarity'] = dist_df_pos['cosine_similarity'].rank(ascending=False, method='min')
        dist_df_pos['rank_distance'] = dist_df_pos['rank_distance'].astype(int)
        dist_df_pos['rank_cosine_similarity'] = dist_df_pos['rank_cosine_similarity'].astype(int)
        return dist_df, dist_df_pos

    return dist_df

In [40]:
def get_unified_player_data(player_name, player_team, df, driver, league):
    df['player_norm'] = df['player'].apply(unidecode)
    player_row = df.loc[df['player_norm'] == player_name].iloc[0]
    player_id_sofa = player_row['sofascore_id']
    player_id_under = player_row['understat_id']
    # options = Options()
    # options.set_preference("permissions.default.image", 2)
    # options.set_preference("devtools.jsonview.enabled", False)

    # driver = webdriver.Firefox(options=options)
    match league:
        case 'Serie A':
            tournament_id = 23
            season_id = 76457
        case 'Premier League':
            tournament_id = 17
            season_id = 76986
        case 'La Liga':
            tournament_id = 8
            season_id = 77559
    
    driver.get(f"https://www.sofascore.com/api/v1/player/{str(int(player_id_sofa))}/unique-tournament/{str(tournament_id)}/season/{str(season_id)}/statistics/overall")
    response_text = driver.find_element("tag name", "body").text
    data = json.loads(response_text)
    player_object = {
        'games': data['statistics']['appearances'],
        'minutes': data['statistics']['minutesPlayed'],
        'nineties': data['statistics']['minutesPlayed']/90,
        'aerials_won_pct': data['statistics']['aerialDuelsWonPercentage'],
        'attempt_assists': data['statistics']['totalAttemptAssist'],
        'ball_recoveries': data['statistics']['ballRecovery'],
        'big_chances_created': data['statistics']['bigChancesCreated'],
        'big_chances_missed': data['statistics']['bigChancesMissed'],
        'blocked_shots': data['statistics']['blockedShots'],
        'chipped_passes': data['statistics']['accurateChippedPasses'],
        'chipped_passes_total': data['statistics']['totalChippedPasses'],
        'clearances': data['statistics']['clearances'],
        'crosses_pct': data['statistics']['accurateCrossesPercentage'],
        'crosses': data['statistics']['totalCross'],
        'dispossessed': data['statistics']['dispossessed'],
        'dribbled_past': data['statistics']['dribbledPast'],
        'dribbled_succ': data['statistics']['successfulDribbles'],
        'dribbles_pct': data['statistics']['successfulDribblesPercentage'],
        # 'duels_won_pct': data['statistics']['totalDuelsWonPercentage'],
        
        'fouled': data['statistics']['wasFouled'],
        'fouls': data['statistics']['fouls'],
        'goal_conversion_pct': data['statistics']['goalConversionPercentage'],
        'ground_duels_won_pct': data['statistics']['groundDuelsWonPercentage'],
        'interceptions': data['statistics']['interceptions'],
        'key_passes': data['statistics']['keyPasses'],
        'long_balls_pct': data['statistics']['accurateLongBallsPercentage'],
        'long_balls': data['statistics']['totalLongBalls'],
        'pass_completion_pct': data['statistics']['accuratePassesPercentage'],
        'pass_final_third': data['statistics']['accurateFinalThirdPasses'],
        'passes': data['statistics']['totalPasses'],
        'possession_lost': data['statistics']['possessionLost'],
        'possession_won_att_third': data['statistics']['possessionWonAttThird'],
        'shots_inside_box': data['statistics']['shotsFromInsideTheBox'],
        'shots_on_target': data['statistics']['shotsOnTarget'],
        'shots_outside_box': data['statistics']['shotsFromOutsideTheBox'],
        'shots': data['statistics']['totalShots'],
        'tackles': data['statistics']['tackles'],
        # 'tackles_won_pct': data['statistics']['tacklesWonPercentage'],
        
        'touches': data['statistics']['touches']
    }

    if 'kilometersCovered' in data['statistics']:
        player_object['kilometers'] = data['statistics']['kilometersCovered']
        player_object['sprints'] = data['statistics']['numberOfSprints']
    else:
        player_object['kilometers'] = 0
        player_object['sprints'] = 0

    player_object['attempt_assists_per90'] = player_object['attempt_assists']/player_object['nineties']
    player_object['ball_recoveries_per90'] = player_object['ball_recoveries']/player_object['nineties']
    player_object['big_chances_created_per90'] = player_object['big_chances_created']/player_object['nineties']
    player_object['big_chances_missed_per90'] = player_object['big_chances_missed']/player_object['nineties']
    player_object['blocked_shots_per90'] = player_object['blocked_shots']/player_object['nineties']
    player_object['chipped_passes_per90'] = player_object['chipped_passes']/player_object['nineties']
    # player_object['chipped_passes_pct'] = player_object['chipped_passes']/player_object['chipped_passes_total']
    if player_object["chipped_passes_total"] != 0:
        player_object["chipped_passes_pct"] = (
            player_object["chipped_passes"]
            / player_object["chipped_passes_total"]
        )
    else:
        player_object["chipped_passes_pct"] = 0
    player_object['clearances_per90'] = player_object['clearances']/player_object['nineties']
    player_object['crosses_per90'] = player_object['crosses']/player_object['nineties']
    player_object['dispossessed_per90'] = player_object['dispossessed']/player_object['nineties']
    player_object['dribbled_past_per90'] = player_object['dribbled_past']/player_object['nineties']
    # player_object['dribbles'] = int(player_object['dribbled_succ']/(player_object['dribbles_pct']/100))
    if player_object["dribbled_succ"] != 0:
        player_object["dribbles"] = (
            int(player_object['dribbled_succ']/(player_object['dribbles_pct']/100))
        )
    else:
        player_object["dribbles"] = 0
    player_object['dribbles_per90'] = player_object['dribbles']/player_object['nineties']
    player_object['fouled_per90'] = player_object['fouled']/player_object['nineties']
    player_object['fouls_per90'] = player_object['fouls']/player_object['nineties']
    player_object['interceptions_per90'] = player_object['interceptions']/player_object['nineties']
    player_object['key_passes_per90'] = player_object['key_passes']/player_object['nineties']
    player_object['long_balls_per90'] = player_object['long_balls']/player_object['nineties']
    player_object['pass_final_third_per90'] = player_object['pass_final_third']/player_object['nineties']
    player_object['passes_per90'] = player_object['passes']/player_object['nineties']
    player_object['possession_lost_per90'] = player_object['possession_lost']/player_object['nineties']
    player_object['possession_won_att_third_per90'] = player_object['possession_won_att_third']/player_object['nineties']
    player_object['shots_inside_box_per90'] = player_object['shots_inside_box']/player_object['nineties']
    player_object['shots_on_target_per90'] = player_object['shots_on_target']/player_object['nineties']
    # player_object['shots_on_target_pct'] = player_object['shots_on_target']/player_object['shots']
    if player_object["shots"] != 0:
        player_object["shots_on_target_pct"] = (
            player_object["shots_on_target"]
            / player_object["shots"]
        )
    else:
        player_object["shots_on_target_pct"] = 0
    player_object['shots_outside_box_per90'] = player_object['shots_outside_box']/player_object['nineties']
    player_object['shots_per90'] = player_object['shots']/player_object['nineties']
    player_object['tackles_per90'] = player_object['tackles']/player_object['nineties']
    player_object['touches_per90'] = player_object['touches']/player_object['nineties']

    player_object['kilometers_per90'] = player_object['kilometers']/player_object['nineties']
    player_object['sprints_per90'] = player_object['sprints']/player_object['nineties']

    url = "https://understat.com/getPlayerData/" + str(int(player_id_under))

    headers = {
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Referer": "https://understat.com/player/189",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/153.0.0.0 Safari/537.36",
        "X-Requested-With": "XMLHttpRequest",
    }

    response = requests.get(url, headers=headers)


    data_under = json.loads(response.text)
    player_object['player'] = data_under['player']['name']
    player_object['position'] = data_under['player']['favorite_position']
    seasons = data_under['groups']['season']
    for stats in seasons:
        if stats['season'] == '2025':
            # print(stats)
            player_object['xA'] = float(stats['xA'])
            # player_object['team'] = stats['team']
            player_object['npxG'] = float(stats['npxG'])
            player_object['xGChain'] = float(stats['xGChain'])
            player_object['xGBuildup'] = float(stats['xGBuildup'])

    player_object['npxG_per90'] = player_object['npxG']/player_object['nineties']
    player_object['xA_per90'] = player_object['xA']/player_object['nineties']
    player_object['xGChain_per90'] = player_object['xGChain']/player_object['nineties']
    player_object['xGBuildup_per90'] = player_object['xGBuildup']/player_object['nineties']
    # player_object['npxG_per_shot'] = player_object['npxG']/player_object['shots']
    if player_object["shots"] != 0:
        player_object["npxG_per_shot"] = (
            player_object["npxG"]
            / player_object["shots"]
        )
    else:
        player_object["npxG_per_shot"] = 0


    shots = pd.DataFrame(data_under['shots'])
    if len(shots) > 0:

        shots = shots.loc[shots['season'] == '2025']

        LENGTH = 105
        WIDTH = 68
        goal_x = LENGTH
        goal_y = WIDTH / 2

        shots["X"] = pd.to_numeric(shots["X"], errors="coerce")
        shots["Y"] = pd.to_numeric(shots["Y"], errors="coerce")
        shots["X_m"] = shots["X"] * LENGTH
        shots["Y_m"] = shots["Y"] * WIDTH

        shots["distance_m"] = np.sqrt(
            (shots["X_m"] - goal_x)**2 +
            (shots["Y_m"] - goal_y)**2
        )
        avg_distance = shots['distance_m'].mean()
        player_object['avg_shot_distance'] = avg_distance
    else:
        player_object['avg_shot_distance'] = 0
    player_object['team'] = player_team
    player_object['league'] = league
    player_df = pd.DataFrame(player_object, index=[0])
    # driver.quit()
    # player_df.to_csv('player_stats.csv', index=False, header=True)
    # dribbles_pct


    # player_df = player_df.drop(columns=['chipped_passes_total', 'attempt_assists', 'ball_recoveries', 'big_chances_created', 'big_chances_missed', 'blocked_shots', 'chipped_passes', 'clearances', 'crosses', 'dispossessed', 'dribbled_past', 'dribbled_succ', 'dribbles', 'fouled', 'fouls', 'interceptions', 'key_passes', 'long_balls', 'npxG', 'pass_final_third', 'passes', 'possession_lost', 'possession_won_att_third', 'shots', 'shots_inside_box', 'shots_on_target', 'shots_outside_box', 'tackles', 'touches', 'xA', 'xGBuildup', 'xGChain'])
    player_df = player_df.drop(columns=['chipped_passes_total', 'attempt_assists', 'ball_recoveries', 'big_chances_created', 'big_chances_missed', 'blocked_shots', 'chipped_passes', 'clearances', 'crosses', 'dispossessed', 'dribbled_past', 'dribbled_succ', 'dribbles', 'fouled', 'fouls', 'interceptions', 'key_passes', 'long_balls', 'npxG', 'pass_final_third', 'passes', 'possession_lost', 'possession_won_att_third', 'shots', 'shots_inside_box', 'shots_on_target', 'shots_outside_box', 'tackles', 'touches', 'xA', 'xGBuildup', 'xGChain', 'kilometers', 'sprints'])
    # player_df = player_df[['player', 'league', 'team', 'position', 'games', 'minutes', 'nineties', 'aerials_won_pct', 'attempt_assists_per90', 'avg_shot_distance', 'ball_recoveries_per90', 'big_chances_created_per90', 'big_chances_missed_per90', 'blocked_shots_per90', 'chipped_passes_pct', 'chipped_passes_per90', 'clearances_per90', 'crosses_pct', 'crosses_per90', 'dispossessed_per90', 'dribbled_past_per90', 'dribbles_pct', 'dribbles_per90', 'duels_won_pct', 'fouled_per90', 'fouls_per90', 'goal_conversion_pct', 'ground_duels_won_pct', 'interceptions_per90', 'key_passes_per90', 'long_balls_pct', 'long_balls_per90', 'npxG_per90', 'npxG_per_shot', 'pass_completion_pct', 'pass_final_third_per90', 'passes_per90', 'possession_lost_per90', 'possession_won_att_third_per90', 'shots_inside_box_per90', 'shots_on_target_pct', 'shots_on_target_per90', 'shots_outside_box_per90', 'shots_per90', 'tackles_per90', 'tackles_won_pct', 'touches_per90', 'xA_per90', 'xGBuildup_per90', 'xGChain_per90']]
    player_df = player_df[['player', 'league', 'team', 'position', 'games', 'minutes', 'nineties', 'aerials_won_pct', 'attempt_assists_per90', 'avg_shot_distance', 'ball_recoveries_per90', 'big_chances_created_per90', 'big_chances_missed_per90', 'blocked_shots_per90', 'chipped_passes_pct', 'chipped_passes_per90', 'clearances_per90', 'crosses_pct', 'crosses_per90', 'dispossessed_per90', 'dribbled_past_per90', 'dribbles_pct', 'dribbles_per90', 'fouled_per90', 'fouls_per90', 'goal_conversion_pct', 'ground_duels_won_pct', 'interceptions_per90', 'key_passes_per90', 'kilometers_per90', 'long_balls_pct', 'long_balls_per90', 'npxG_per90', 'npxG_per_shot', 'pass_completion_pct', 'pass_final_third_per90', 'passes_per90', 'possession_lost_per90', 'possession_won_att_third_per90', 'shots_inside_box_per90', 'shots_on_target_pct', 'shots_on_target_per90', 'shots_outside_box_per90', 'shots_per90', 'sprints_per90', 'tackles_per90', 'touches_per90', 'xA_per90', 'xGBuildup_per90', 'xGChain_per90']]
    return player_df

In [41]:
def build_stats_sofa_under(league):
    match league:
        case 'Serie A':
            teams = ['Atalanta', 'Bologna', 'Cagliari', 'Cremonese', 'Como', 'Fiorentina', 'Genoa', 'Inter', 'Juventus', 'Lazio', 'Lecce', 'Milan', 'Napoli', 'Parma', 'Pisa', 'Roma', 'Sassuolo', 'Torino', 'Udinese', 'Verona']
        case 'Premier League':
            teams = ['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Leeds', 'Liverpool', 'Manchester City', 'Manchester United', 'Newcastle', 'Nottingham', 'Sunderland', 'Tottenham', 'West Ham', 'Wolverhampton']
        case 'La Liga':
            teams = ['Barcelona', 'Real Madrid', 'Villarreal', 'Atletico Madrid', 'Betis', 'Celta Vigo', 'Getafe', 'Rayo Vallecano', 'Valencia', 'Real Sociedad', 'Espanyol', 'Elche', 'Athletic Club', 'Sevilla', 'Alaves', 'Elche', 'Levante', 'Osasuna', 'Mallorca', 'Girona', 'Real Oviedo']

    df = pd.read_csv('unified_player_stats.csv')
    df = df.loc[df['season'] == '2025-26']
    pattern = "|".join(map(re.escape, teams))

    options = Options()
    options.set_preference("permissions.default.image", 2)
    options.set_preference("devtools.jsonview.enabled", False)

    driver = webdriver.Firefox(options=options)
    driver.get("https://www.sofascore.com/")

    filtered_df = df[df["team"].str.contains(pattern, case=False, na=False)]
    filtered_df['player_norm'] = filtered_df['player'].apply(unidecode)
    filtered_df = filtered_df.reset_index(drop=True)
    for i in tqdm(filtered_df.index, "Fetching Player Stats"):
        player_row = filtered_df.loc[i]
        player_name = player_row['player_norm']
        player_team = player_row['team']
        player_df = get_unified_player_data(player_name, player_team, filtered_df, driver, league)
        # if i == 0:
        #     # player_df.to_csv('43to7/stats_sofa_under.csv', header=True, index=False)
        #     player_df.to_csv('43to7/stats_sofa_under_kmsp.csv', header=True, index=False)
        # else:
            # player_df.to_csv('43to7/stats_sofa_under.csv', header=False, index=False, mode='a')
        player_df.to_csv('43to7/stats_sofa_under_kmsp.csv', header=False, index=False, mode='a')
    # filtered_df

In [26]:
# drop team duplicates
df = pd.read_csv('43to7/stats_sofa_under_kmsp.csv')
# df = df.drop(columns=['team'])
df[df.duplicated(subset='player', keep=False)].sort_values(by=['player'])

cols = [c for c in df.columns if c != "team"]

df = (
    df.assign(team_len=df["team"].str.len())
      .sort_values("team_len")
      .drop_duplicates(subset=cols, keep="first")
      .drop(columns="team_len")
)

df = df.sort_values(by=['league', 'player']).reset_index(drop=True)
df.to_csv('43to7/stats_sofa_under.csv', index=False, header=True)

In [29]:
df = pd.read_csv('43to7/stats_sofa_under.csv')
df["avg_shot_distance"] = df["avg_shot_distance"].fillna(0)
df.to_csv('43to7/stats_sofa_under.csv', index=False, header=True)

In [42]:
build_stats_sofa_under('La Liga')

C:\Users\an.capobianco\AppData\Local\Temp\ipykernel_23340\4055072720.py:10: DtypeWarning: Columns (0: pos) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('unified_player_stats.csv')
C:\Users\an.capobianco\AppData\Local\Temp\ipykernel_23340\4055072720.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  filtered_df['player_norm'] = filtered_df['player'].apply(unidecode)
Fetching Player Stats:  15%|█▍        | 92/628 [01:08<06:40,  1.34it/s]


KeyError: 'statistics'

In [ ]:
# old dataset
build_stats()
df = build_df()
df_standard = standardize_df(df, sofa_under=False, minutes=100)
df_pca, cont_df, loadings = apply_pca(df_standard, sofa_under=False)
# df_pca.head()
print(loadings['loading_strength'].sort_values(ascending=False))
# for i in range(len(df.columns)):
#     print(i, sorted(df.columns)[i])

In [33]:
df = pd.read_csv('43to7/stats_sofa_under.csv')
df_standard = standardize_df(df, sofa_under=True, minutes=1000)
df_pca, cont_df, loadings = apply_pca(df_standard, sofa_under=True)
# print(loadings['loading_strength'].sort_values(ascending=False))


PCA Explained Variance: [0.2976829  0.16771416 0.0847257  0.0480309  0.04030196 0.03386693
 0.0300094 ]
PCA Explained Variance Sum: 70.23%


In [ ]:
cont_df[['feature', 'cont_6']].sort_values(key=abs,by=['cont_6'], ascending=False).reset_index(drop=True)

In [ ]:
print(cont_df[['feature', 'cont_0']].sort_values(key=abs,by=['cont_0'], ascending=False).head(10))
print(cont_df[['feature', 'cont_1']].sort_values(key=abs,by=['cont_1'], ascending=False).head(10))
print(cont_df[['feature', 'cont_2']].sort_values(key=abs,by=['cont_2'], ascending=False).head(10))
print(cont_df[['feature', 'cont_3']].sort_values(key=abs,by=['cont_3'], ascending=False).head(10))
print(cont_df[['feature', 'cont_4']].sort_values(key=abs,by=['cont_4'], ascending=False).head(10))
print(cont_df[['feature', 'cont_5']].sort_values(key=abs,by=['cont_5'], ascending=False).head(10))
print(cont_df[['feature', 'cont_6']].sort_values(key=abs,by=['cont_6'], ascending=False).head(10))

In [32]:
player_name = "Federico Dimarco"
sim_df, sim_df_pos = get_distances(player_name, df_pca, sofa_under=True)
print(sim_df.head())
print(sim_df_pos.head())

Computing distances: 100%|██████████| 593/593 [00:00<00:00, 765.55it/s]

                 player               team position  distance  \
0      Federico Dimarco              Inter      DML  0.000000   
1       Bruno Fernandes  Manchester United      AMC  3.298849   
2         Mathis Cherki    Manchester City      AMC  4.329969   
3  Charles De Ketelaere           Atalanta      AMC  4.670324   
4       Martin Baturina               Como      AML  5.122886   

   cosine_similarity  rank_distance  rank_cosine_similarity  
0           1.000000              1                       1  
1           0.963129              2                       2  
2           0.939326              3                       3  
3           0.920218              4                       5  
4           0.906509              5                       6  
                 player      team position  distance  cosine_similarity  \
0      Federico Dimarco     Inter      DML  0.000000           1.000000   
20      Nicola Zalewski  Atalanta      DML  7.093662           0.837221   
44      Andr

In [ ]:
df = pd.read_csv('stats_reduced.csv')
df = df.drop(columns=['player', 'player_norm', 'team', 'pos', 'games', 'minutes', 'ninety_s'])
columns = df.columns
corr = df[columns].corr()
corr_pairs = (
    corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )
    .stack()
    .sort_values(key=abs, ascending=False)
)

print(corr_pairs.head(30))